# 02 - Model Training & Comparison

**Objective:** Train and compare multiple machine learning models for car price prediction, select the best performer, and save it for deployment.

**Models evaluated:**
1. Linear Regression (baseline)
2. Random Forest Regressor
3. XGBoost Regressor

**Evaluation metrics:** MAE, RMSE, R-squared, 5-fold cross-validation

**Dataset:** CarDekho cleaned dataset (`cleaned_data.csv`) with log-transformed target variable.

---
## 1. Imports & Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("xgboost is required. Install with: pip install xgboost")

# Import project modules
sys.path.insert(0, '..')
from src.feature_engineering import build_pipeline, prepare_data_log, get_feature_columns

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('Setup complete.')

---
## 2. Load Data

In [ ]:
df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Quick sanity check
print(f'Null values:\n{df.isnull().sum()}')
print(f'\nTarget (Selling_Price) stats:\n{df["Selling_Price"].describe()}')

---
## 3. Feature Engineering Pipeline

We use an sklearn `ColumnTransformer` inside a `Pipeline` that:
- **Numeric features** (`Present_Price`, `Kms_Driven`, `Owner`, `Car_Age`): StandardScaler
- **Categorical features** (`Brand`, `Fuel_Type`, `Seller_Type`, `Transmission`): OneHotEncoder

The target variable (`Selling_Price`) is log-transformed via `np.log1p()` to reduce skewness. We invert predictions with `np.expm1()` when evaluating on the original price scale.

In [ ]:
# Prepare features and log-transformed target
X, y = prepare_data_log(df)

print(f'Feature columns: {get_feature_columns()}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\ny (log-transformed) sample:\n{y.head()}')

---
## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Split ratio:  {X_train.shape[0] / len(X):.0%} / {X_test.shape[0] / len(X):.0%}')

---
## 5. Model Training

We train three models, each wrapped in the same preprocessing pipeline. All metrics are computed on the **log-transformed** scale for consistency, plus MAE on the **original price** scale using `np.expm1()`.

In [ ]:
def evaluate_model(pipeline, X_train, X_test, y_train, y_test, model_name):
    """Fit a pipeline, predict, and return metrics dict."""
    # Fit
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    
    # Metrics on log scale
    mae_log = mean_absolute_error(y_test, y_pred_test)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2 = r2_score(y_test, y_pred_test)
    r2_train = r2_score(y_train, y_pred_train)
    
    # MAE on original price scale (Lakhs)
    mae_actual = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred_test))
    
    # 5-fold cross-validation (R2)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    
    metrics = {
        'Model': model_name,
        'MAE (log)': round(mae_log, 4),
        'RMSE (log)': round(rmse_log, 4),
        'R2 Train': round(r2_train, 4),
        'R2 Test': round(r2, 4),
        'MAE (Lakhs)': round(mae_actual, 2),
        'CV R2 Mean': round(cv_scores.mean(), 4),
        'CV R2 Std': round(cv_scores.std(), 4),
    }
    
    print(f'\n{"=" * 50}')
    print(f'  {model_name}')
    print(f'{"=" * 50}')
    print(f'  MAE  (log scale):    {mae_log:.4f}')
    print(f'  RMSE (log scale):    {rmse_log:.4f}')
    print(f'  R2   (train):        {r2_train:.4f}')
    print(f'  R2   (test):         {r2:.4f}')
    print(f'  MAE  (actual Lakhs): {mae_actual:.2f}')
    print(f'  5-Fold CV R2:        {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    
    return metrics, pipeline, y_pred_test

### 5.1 Linear Regression (Baseline)

In [ ]:
lr_pipeline = build_pipeline(LinearRegression())
lr_metrics, lr_pipeline, lr_preds = evaluate_model(
    lr_pipeline, X_train, X_test, y_train, y_test, 'Linear Regression'
)

### 5.2 Random Forest

In [ ]:
rf_pipeline = build_pipeline(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
)
rf_metrics, rf_pipeline, rf_preds = evaluate_model(
    rf_pipeline, X_train, X_test, y_train, y_test, 'Random Forest'
)

### 5.3 XGBoost

In [ ]:
xgb_pipeline = build_pipeline(
    XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
)
xgb_metrics, xgb_pipeline, xgb_preds = evaluate_model(
    xgb_pipeline, X_train, X_test, y_train, y_test, 'XGBoost'
)

---
## 6. Model Comparison Table

In [ ]:
results_df = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics])
results_df = results_df.set_index('Model')

# Highlight the best model
print('Model Comparison (sorted by Test R2):\n')
results_df.sort_values('R2 Test', ascending=False)

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = results_df.index.tolist()
colors = ['#3498db', '#2ecc71', '#e74c3c']

axes[0].barh(models, results_df['R2 Test'], color=colors)
axes[0].set_xlabel('R2 Score')
axes[0].set_title('Test R2 Score')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results_df['R2 Test']):
    axes[0].text(v + 0.01, i, f'{v:.4f}', va='center')

axes[1].barh(models, results_df['RMSE (log)'], color=colors)
axes[1].set_xlabel('RMSE (log scale)')
axes[1].set_title('Test RMSE')
for i, v in enumerate(results_df['RMSE (log)']):
    axes[1].text(v + 0.005, i, f'{v:.4f}', va='center')

axes[2].barh(models, results_df['MAE (Lakhs)'], color=colors)
axes[2].set_xlabel('MAE (Lakhs)')
axes[2].set_title('Test MAE (Actual Prices)')
for i, v in enumerate(results_df['MAE (Lakhs)']):
    axes[2].text(v + 0.05, i, f'{v:.2f}', va='center')

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Best Model Evaluation

Select the best model based on test R2 score and perform detailed evaluation.

In [ ]:
# Select best model
all_results = [lr_metrics, rf_metrics, xgb_metrics]
all_pipelines = [lr_pipeline, rf_pipeline, xgb_pipeline]
all_preds = [lr_preds, rf_preds, xgb_preds]

best_idx = np.argmax([m['R2 Test'] for m in all_results])
best_name = all_results[best_idx]['Model']
best_pipeline = all_pipelines[best_idx]
best_preds = all_preds[best_idx]

print(f'Best model: {best_name}')
print(f'Test R2:    {all_results[best_idx]["R2 Test"]}')
print(f'Test RMSE:  {all_results[best_idx]["RMSE (log)"]}')
print(f'CV R2:      {all_results[best_idx]["CV R2 Mean"]} +/- {all_results[best_idx]["CV R2 Std"]}')

### 7.1 Actual vs Predicted Scatter Plot

In [ ]:
# Convert back to actual prices
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(best_preds)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test_actual, y_pred_actual, alpha=0.6, edgecolors='k', linewidth=0.5)

# Perfect prediction line
max_val = max(y_test_actual.max(), y_pred_actual.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Price (Lakhs)', fontsize=12)
ax.set_ylabel('Predicted Price (Lakhs)', fontsize=12)
ax.set_title(f'{best_name}: Actual vs Predicted Prices', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Residual Plot

In [ ]:
residuals = y_test_actual - y_pred_actual

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_actual, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Price (Lakhs)', fontsize=12)
axes[0].set_ylabel('Residual (Lakhs)', fontsize=12)
axes[0].set_title('Residuals vs Predicted', fontsize=13)

# Residual distribution
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='#3498db')
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (Lakhs)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=13)

plt.suptitle(f'{best_name}: Residual Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/residual_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean residual:   {residuals.mean():.4f} Lakhs')
print(f'Median residual: {residuals.median():.4f} Lakhs')
print(f'Std residual:    {residuals.std():.4f} Lakhs')

### 7.3 Feature Importance

In [ ]:
# Extract feature names after preprocessing
preprocessor = best_pipeline.named_steps['preprocessor']
model = best_pipeline.named_steps['model']

# Get feature names from the fitted preprocessor
try:
    feature_names = preprocessor.get_feature_names_out()
except AttributeError:
    # Fallback for older sklearn
    num_features = preprocessor.transformers_[0][2]
    cat_features = preprocessor.transformers_[1][1].named_steps['onehot'].get_feature_names_out(
        preprocessor.transformers_[1][2]
    )
    feature_names = list(num_features) + list(cat_features)

# Get importances (works for tree-based models)
if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    feat_imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=True)
    
    # Plot top 15 features
    top_n = min(15, len(feat_imp_df))
    plot_df = feat_imp_df.tail(top_n)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(plot_df)), plot_df['Importance'], color='#2ecc71', edgecolor='black')
    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(plot_df['Feature'], fontsize=10)
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'{best_name}: Top {top_n} Feature Importances', fontsize=14)
    plt.tight_layout()
    plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
elif hasattr(model, 'coef_'):
    # Linear model coefficients
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': np.abs(model.coef_)
    }).sort_values('Coefficient', ascending=True)
    
    top_n = min(15, len(coef_df))
    plot_df = coef_df.tail(top_n)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(plot_df)), plot_df['Coefficient'], color='#3498db', edgecolor='black')
    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(plot_df['Feature'], fontsize=10)
    ax.set_xlabel('|Coefficient|', fontsize=12)
    ax.set_title(f'{best_name}: Top {top_n} Feature Coefficients (abs)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

### 7.4 Learning Curves

Learning curves show how training and validation scores change as the training set size increases. This helps diagnose whether the model suffers from high bias (underfitting) or high variance (overfitting).

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='r2', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Score')
ax.plot(train_sizes, val_mean, 'o-', color='orange', label='Validation Score')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('R2 Score', fontsize=12)
ax.set_title(f'{best_name}: Learning Curves', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Save Best Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model_path = '../models/best_model.pkl'
joblib.dump(best_pipeline, model_path)
print(f'Best model ({best_name}) saved to: {model_path}')

# Verify by loading
loaded = joblib.load(model_path)
verify_preds = loaded.predict(X_test[:3])
print(f'Verification predictions (log): {verify_preds}')
print(f'Verification predictions (Lakhs): {np.expm1(verify_preds)}')
print('Model saved and verified successfully.')

---
## 9. Conclusion

### Summary of Results

In [ ]:
# Final comparison table
print('Final Model Comparison')
print('=' * 70)
display(results_df.sort_values('R2 Test', ascending=False))

print(f'\nBest Performing Model: {best_name}')
print(f'  - Test R2 Score: {all_results[best_idx]["R2 Test"]}')
print(f'  - Test RMSE (log): {all_results[best_idx]["RMSE (log)"]}')
print(f'  - MAE on actual prices: {all_results[best_idx]["MAE (Lakhs)"]} Lakhs')
print(f'  - 5-Fold CV R2: {all_results[best_idx]["CV R2 Mean"]} (+/- {all_results[best_idx]["CV R2 Std"]})')
print(f'\nThe {best_name} model generalises well with consistent cross-validation')
print(f'scores and the lowest prediction error on unseen data.')
print(f'The model has been saved to: ../models/best_model.pkl')

### Key Takeaways

1. **Linear Regression** serves as a reasonable baseline but struggles to capture non-linear relationships in the data.
2. **Random Forest** and **XGBoost** both significantly outperform the linear baseline, capturing complex feature interactions.
3. The **log transformation** on the target variable helps stabilise variance and improves model performance across all methods.
4. **Present_Price** and **Car_Age** are consistently the most important predictors of selling price.
5. The best model achieves strong generalisation as shown by the learning curves and cross-validation stability.

### Next Steps

- Deploy the best model via the Flask API for real-time predictions.
- Monitor prediction performance on new data over time.
- Consider hyperparameter tuning (GridSearchCV / Optuna) for further marginal gains.